In [3]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Montar Drive
drive.mount('/content/drive', force_remount=True)
folder_path = '/content/drive/MyDrive/bt/'
file_name = 'ARC_LIVE_Boom 1000 Index.csv'
full_path = os.path.join(folder_path, file_name)

def analyze_big_data_v2(path):
    print(f"🚀 Iniciando Minería Avanzada en: {file_name}")

    # Nombres de las columnas del EA
    column_names = ["Time", "Price", "NodeM2", "NodeH12", "Upsilon", "Energy", "ConfUP"]
    spike_events = []
    chunk_size = 100000
    total_processed = 0

    # --- DIAGNÓSTICO: Detectar separador ---
    with open(path, 'r') as f:
        first_line = f.readline()
        print(f"🔍 Muestra de datos: {first_line[:100]}")
        # Detectamos si es coma, punto y coma o tabulación
        if ';' in first_line: separator = ';'
        elif '\t' in first_line: separator = '\t'
        else: separator = ','
    print(f"✅ Separador detectado: '{separator}'")

    try:
        # Leemos con el separador correcto y saltando líneas erróneas
        reader = pd.read_csv(path,
                             sep=separator,
                             names=column_names,
                             header=0,
                             chunksize=chunk_size,
                             on_bad_lines='skip', # Ignora filas corruptas
                             engine='c')

        for i, chunk in enumerate(reader):
            # Asegurar que los datos sean numéricos (limpieza de emergencia)
            chunk['Price'] = pd.to_numeric(chunk['Price'], errors='coerce')
            chunk['Energy'] = pd.to_numeric(chunk['Energy'], errors='coerce')

            # A. Detectar Spike
            chunk['Price_Change'] = chunk['Price'].diff()

            # B. Filtro de Interés: Spikes > 2.0 o Energía > 15,000
            # Usamos .query para mayor velocidad
            critical = chunk.query('Price_Change > 2.0 or Energy > 15000').copy()

            if not critical.empty:
                spike_events.append(critical)

            total_processed += len(chunk)
            if i % 10 == 0:
                print(f"⏳ Procesadas {total_processed:,} filas...")

        if spike_events:
            df_final = pd.concat(spike_events)

            print("\n--- 🔱 INFORME DE MINERÍA ARC v43.0 ---")
            print(f"📊 Total de eventos clave: {len(df_final)}")

            # Estadísticas Maestras
            big_spikes = df_final[df_final['Price_Change'] > 8.0]
            if not big_spikes.empty:
                print(f"🔥 Energía Promedio en éxitos: {big_spikes['Energy'].mean():.0f}")
                print(f"🌊 Tensión (Upsilon) en éxitos: {big_spikes['Upsilon'].mean():.4f}")
                print(f"🎯 Nodo Promedio en éxitos: {big_spikes['NodeM2'].mean():.2f}")

            # Guardar extracto para análisis fino
            output = folder_path + 'ARC_SIGNATURE_FINAL.csv'
            df_final.to_csv(output, index=False)
            print(f"\n💾 Archivo de firmas guardado: {output}")
            return df_final
        else:
            print("🛑 No se encontraron eventos críticos. ¿El archivo tiene datos?")
            return None

    except Exception as e:
        print(f"❌ Error durante el proceso: {e}")

# Ejecutar
df_signatures = analyze_big_data_v2(full_path)

Mounted at /content/drive
🚀 Iniciando Minería Avanzada en: ARC_LIVE_Boom 1000 Index.csv
🔍 Muestra de datos: Time	Price	NodeM2	NodeH12	Upsilon	Energy	ConfUP

✅ Separador detectado: '	'
⏳ Procesadas 100,000 filas...
⏳ Procesadas 1,100,000 filas...
⏳ Procesadas 2,100,000 filas...
⏳ Procesadas 3,100,000 filas...
⏳ Procesadas 4,100,000 filas...
⏳ Procesadas 5,100,000 filas...
⏳ Procesadas 6,100,000 filas...
⏳ Procesadas 7,100,000 filas...
⏳ Procesadas 8,100,000 filas...
⏳ Procesadas 9,100,000 filas...
⏳ Procesadas 10,100,000 filas...
⏳ Procesadas 11,100,000 filas...
⏳ Procesadas 12,100,000 filas...
⏳ Procesadas 13,100,000 filas...
⏳ Procesadas 14,100,000 filas...
⏳ Procesadas 15,100,000 filas...

--- 🔱 INFORME DE MINERÍA ARC v43.0 ---
📊 Total de eventos clave: 13914176
🔥 Energía Promedio en éxitos: 3478770
🌊 Tensión (Upsilon) en éxitos: 61.6836
🎯 Nodo Promedio en éxitos: 107.17

💾 Archivo de firmas guardado: /content/drive/MyDrive/bt/ARC_SIGNATURE_FINAL.csv


In [5]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Conexión
drive.mount('/content/drive', force_remount=True)
full_path = '/content/drive/MyDrive/bt/ARC_GOLD_BIO_DATA.csv'

def find_ignition_signature(path):
    print("🛰️ Analizando ADN de Ignición en 2GB...")
    chunk_size = 100000
    ignitions = []

    # Nombres de columnas según tu último EA
    cols = ["Time", "Price", "NodeM2", "NodeH12", "Upsilon", "Energy", "ConfUP"]

    try:
        reader = pd.read_csv(path, sep=';', names=cols, header=0, chunksize=chunk_size)

        for chunk in reader:
            # Convertir a numérico por seguridad
            chunk['Price'] = pd.to_numeric(chunk['Price'], errors='coerce')
            chunk['Energy'] = pd.to_numeric(chunk['Energy'], errors='coerce')

            # Detectar el inicio del Spike
            chunk['Jump'] = chunk['Price'].diff()

            # Filtramos momentos donde hubo un salto real (> 3 pips)
            # Queremos ver los datos de LA FILA ANTERIOR al salto (la ignición)
            spikes_idx = chunk[chunk['Jump'] > 3.0].index

            # Retrocedemos 1 fila para ver el estado previo al estallido
            ignition_rows = chunk.loc[spikes_idx - 1].dropna()

            if not ignition_rows.empty:
                ignitions.append(ignition_rows)

        if ignitions:
            df_ign = pd.concat(ignitions)

            print("\n--- 🔱 FIRMA REAL DE IGNICIÓN (ESTADO PRE-SPIKE) ---")
            # Usamos el Percentil 10 para ver el "Mínimo Necesario" para ganar
            print(f"🔥 Energía Mínima (Percentil 10): {df_ign['Energy'].quantile(0.10):.0f}")
            print(f"🔥 Energía Recomendada (Mediana): {df_ign['Energy'].median():.0f}")
            print(f"🎯 Nodo Mínimo (Percentil 10): {df_ign['NodeM2'].quantile(0.10):.2f}")
            print(f"🌊 Upsilon Máximo (Percentil 90): {df_ign['Upsilon'].quantile(0.90):.4f}")

            return df_ign
        else:
            print("🛑 No se detectaron igniciones. Revisa el separador del CSV.")
    except Exception as e:
        print(f"❌ Error: {e}")

# Ejecutar
df_results = find_ignition_signature(full_path)

Mounted at /content/drive
🛰️ Analizando ADN de Ignición en 2GB...

--- 🔱 FIRMA REAL DE IGNICIÓN (ESTADO PRE-SPIKE) ---
🔥 Energía Mínima (Percentil 10): 13635
🔥 Energía Recomendada (Mediana): 75300
🎯 Nodo Mínimo (Percentil 10): 10.94
🌊 Upsilon Máximo (Percentil 90): 0.6447


In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURACIÓN CANÓNICA ARC v3.4 ---
FILE_PATH = 'ARC_GOLD_DATA_Boom 1000 Index_1095000.csv'

def revisar_malla_gold(path):
    print(f"🛰️ Iniciando revisión física de: {path}")

    try:
        # Autodetección de delimitador y limpieza de líneas corruptas
        # El Canon establece: Timestamp, Price, Node, Sexto, Energy, Upsilon, Speed (o Symbol)
        df = pd.read_csv(path, sep=None, engine='python', on_bad_lines='skip')

        # Renombrar columnas para alineación con Tesis ARC
        # Ajustamos dinámicamente según lo que encuentre el motor
        columnas_canon = ['Timestamp', 'Price', 'Node', 'Sexto', 'Energy', 'Upsilon']
        df.columns = columnas_canon + list(df.columns[len(columnas_canon):])

        print(f"✅ Sincronización exitosa. Registros cargados: {len(df):,}")

        # 1. Análisis de Tensión Superficial (Upsilon)
        # Buscamos el punto de Saturación: Upsilon < 0.25
        saturacion = df[df['Upsilon'] < 0.25]
        print(f"⚠️ Eventos de Saturación (Upsilon < 0.25): {len(saturacion):,}")

        # 2. Análisis de la Zona Gloria (72-82)
        zona_gloria = df[(df['Node'] >= 72) & (df['Node'] <= 82)]
        prob_ignicion = (len(zona_gloria[zona_gloria['Upsilon'] < 0.25]) / len(zona_gloria)) * 100
        print(f"🎯 Densidad en Zona Gloria: {len(zona_gloria):,} registros")
        print(f"🔥 Probabilidad de Ignición en Nodos 72-82: {prob_ignicion:.2f}%")

        # 3. Detección de Spikes (Diferencial de Precio)
        # Un spike se define como una variación súbita > 2.0 en el índice Boom
        df['Delta_P'] = df['Price'].diff()
        spikes = df[df['Delta_P'] > 2.0]

        print(f"🚀 Spikes detectados en la muestra: {len(spikes)}")

        return df, spikes

    except Exception as e:
        print(f"❌ Error Crítico en la Malla: {str(e)}")
        return None, None

# Ejecución de Auditoría
df_gold, lista_spikes = revisar_malla_gold(FILE_PATH)

🛰️ Iniciando revisión física de: ARC_GOLD_DATA_Boom 1000 Index_1095000.csv
❌ Error Crítico en la Malla: [Errno 2] No such file or directory: 'ARC_GOLD_DATA_Boom 1000 Index_1095000.csv'
